In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import CategoricalNB

df = pd.read_csv("agaricus-lepiota.data", header=None)

# First column = class label
y = df.iloc[:, 0]          # 'e' or 'p'
X = df.iloc[:, 1:]

# Train/test split (75/25)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# ============================================================
# Custom Naive Bayes (with Laplace smoothing)
# ============================================================

class MyNaiveBayes:
    def __init__(self, alpha=1):
        self.alpha = alpha

    def fit(self, X, y):
        self.classes = y.unique()
        self.class_priors = {}
        self.feature_probs = {}
        self.feature_values = {}

        n = len(y)

        # store possible values
        for col in X.columns:
            self.feature_values[col] = X[col].unique()

        # priors
        for c in self.classes:
            self.class_priors[c] = np.sum(y == c) / n

        # conditional probabilities
        for c in self.classes:
            X_c = X[y == c]
            self.feature_probs[c] = {}

            for col in X.columns:
                self.feature_probs[c][col] = {}
                values = self.feature_values[col]
                k = len(values)

                for val in values:
                    count = np.sum(X_c[col] == val)
                    prob = (count + self.alpha) / (len(X_c) + self.alpha * k)
                    self.feature_probs[c][col][val] = prob

    def predict(self, X):
        preds = []

        for i in range(len(X)):
            row = X.iloc[i]
            scores = {}

            for c in self.classes:
                log_prob = np.log(self.class_priors[c])

                for col in X.columns:
                    val = row[col]
                    prob = self.feature_probs[c][col].get(val, 1e-9)
                    log_prob += np.log(prob)

                scores[c] = log_prob

            preds.append(max(scores, key=scores.get))

        return np.array(preds)

# Train custom model
my_nb = MyNaiveBayes(alpha=1)
my_nb.fit(X_train, y_train)

# Predict
y_pred_custom = my_nb.predict(X_test)

# Metrics (treat 'p' as positive class)
acc_custom = accuracy_score(y_test, y_pred_custom)
prec_custom = precision_score(y_test, y_pred_custom, pos_label='p')
rec_custom = recall_score(y_test, y_pred_custom, pos_label='p')
f1_custom = f1_score(y_test, y_pred_custom, pos_label='p')

print("Custom Naive Bayes:")
print(f"Accuracy:  {acc_custom:.4f}")
print(f"Precision: {prec_custom:.4f}")
print(f"Recall:    {rec_custom:.4f}")
print(f"F1 Score:  {f1_custom:.4f}")

# ============================================================
# Package Naive Bayes (CategoricalNB)
# ============================================================

# Encode categorical features
X_train_enc = X_train.copy()
X_test_enc = X_test.copy()

encoders = {}
for col in X_train.columns:
    le = LabelEncoder()
    X_train_enc[col] = le.fit_transform(X_train[col])
    X_test_enc[col] = le.transform(X_test[col])
    encoders[col] = le

# Encode labels
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc = label_encoder.transform(y_test)

# Train package model
pkg_nb = CategoricalNB(alpha=1.0)
pkg_nb.fit(X_train_enc, y_train_enc)

# Predict
y_pred_pkg_enc = pkg_nb.predict(X_test_enc)
y_pred_pkg = label_encoder.inverse_transform(y_pred_pkg_enc)

# Metrics
acc_pkg = accuracy_score(y_test, y_pred_pkg)
prec_pkg = precision_score(y_test, y_pred_pkg, pos_label='p')
rec_pkg = recall_score(y_test, y_pred_pkg, pos_label='p')
f1_pkg = f1_score(y_test, y_pred_pkg, pos_label='p')

print("\nPackage Naive Bayes:")
print(f"Accuracy:  {acc_pkg:.4f}")
print(f"Precision: {prec_pkg:.4f}")
print(f"Recall:    {rec_pkg:.4f}")
print(f"F1 Score:  {f1_pkg:.4f}")

Custom Naive Bayes:
Accuracy:  0.9527
Precision: 0.9911
Recall:    0.9101
F1 Score:  0.9489

Package Naive Bayes:
Accuracy:  0.9527
Precision: 0.9911
Recall:    0.9101
F1 Score:  0.9489
